In [8]:
import numpy as np
import pandas as pd
from pathlib import Path

In [9]:
dir_path = Path("./Datasets")
target_csv = dir_path / "analytical_base_table.csv"
target_parquet = dir_path / "analytical_base_table.parquet"

if dir_path.exists() and dir_path.is_dir():
    if target_csv.exists():
        abt = pd.read_csv(target_csv)
    elif target_parquet.exists():
        abt = pd.read_parquet(target_parquet)
    else:
        print("Directory exists, but the specific file was not found.")
else:
    print("Directory does not exist.")

## **Null Statistics**


In [10]:
null_counts = abt.isnull().sum()
null_percent = (abt.isnull().sum() / len(abt)) * 100

null_stats = pd.DataFrame(
    {
        "Feature": abt.columns,
        "Null Count": null_counts.values,
        "Null Percentage (%)": null_percent.round(2).values,
    }
).sort_values(by="Null Count", ascending=False)

print("Null Statistics")
null_stats

Null Statistics


,Feature,Null Count,Null Percentage (%)
0,station,0,0.0
1,lat,0,0.0
2,datetime_utc,0,0.0
3,pm25 in µg/m^3,0,0.0
4,lon,0,0.0
5,city,0,0.0
6,province,0,0.0
7,population,0,0.0
8,area_in_sq.km,0,0.0
9,density_persons/sqkm,0,0.0


## **Numerical Feature Statistics**


In [11]:
numeric_stats = abt.describe(include=[np.number],percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).T
print("\nNumerical Feature Statistics")
numeric_stats


Numerical Feature Statistics


,count,mean,std,min,5%,10%,25%,50%,75%,90%,95%,max
lat,1027171.0,14.297758,1.167600,7.143752,10.73537,14.37853,14.54570,14.58455,14.64443,14.71405,14.769440,16.49742
pm25 in µg/m^3,1027171.0,19.414052,28.260038,0.000000,2.03000,3.87000,7.80000,14.80000,24.08000,34.28000,44.089866,987.89000
lon,1027171.0,121.167812,0.778866,120.490300,120.59750,120.75050,120.98362,121.01877,121.05675,121.09803,122.548820,125.64968


## **Categorical Feature Statistics**


In [12]:
categorical_stats = abt.describe(include=["object", "category", "str"]).T
print("\nCategorical Feature Statistics")
categorical_stats


Categorical Feature Statistics


,count,unique,top,freq
station,1027171,97,3S Center Bagbaguin,17000
datetime_utc,1027171,897758,2026-09-04 01:00:00+00:00,20
city,1027171,37,Manila,103551
province,1027171,14,NCR,821222
population,1027171,14,"14,001,751",821222
area_in_sq.km,1027171,14,619.54,821222
density_persons/sqkm,1027171,14,"21,765",821222


## **Grouped Statistics by Province**


In [13]:
percentiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
quantile_funcs = [
    (f"p{int(p*100):02d}", (lambda q: lambda x: x.quantile(q))(p))
    for p in percentiles
]

print("\nGrouped Statistics by Province")
grouped_stats = abt.groupby("province")[["lat", "lon", "pm25 in µg/m^3"]].agg(["mean","min","max"] + quantile_funcs).T
grouped_stats


Grouped Statistics by Province


province                 Bataan     Benguet  Camarines Norte       Capiz  \
lat            mean   14.625148   16.497420        14.115700   11.430320   
               min    14.436000   16.497420        14.115700   11.430320   
               max    14.769440   16.497420        14.115700   11.430320   
               p05    14.515000   16.497420        14.115700   11.430320   
               p10    14.515000   16.497420        14.115700   11.430320   
               p25    14.515000   16.497420        14.115700   11.430320   
               p50    14.550700   16.497420        14.115700   11.430320   
               p75    14.677590   16.497420        14.115700   11.430320   
               p90    14.769440   16.497420        14.115700   11.430320   
               p95    14.769440   16.497420        14.115700   11.430320   
lon            mean  120.565661  120.653413       122.955100  122.926229   
               min   120.490300  120.653413       122.955100  122.926229   
               max   120.609000  120.653413       122.955100  122.926229   
               p05   120.518200  120.653413       122.955100  122.926229   
               p10   120.518200  120.653413       122.955100  122.926229   
               p25   120.518200  120.653413       122.955100  122.926229   
               p50   120.542780  120.653413       122.955100  122.926229   
               p75   120.597500  120.653413       122.955100  122.926229   
               p90   120.609000  120.653413       122.955100  122.926229   
               p95   120.609000  120.653413       122.955100  122.926229   
pm25 in µg/m^3 mean   10.622870   16.794275        31.434669   32.040528   
               min     0.000000    0.000000         0.100000    0.000000   
               max   144.970000  858.000000       125.700000  537.100000   
               p05     1.470000    0.100000         8.090000    2.300000   
               p10     2.550000    0.500000        11.500000    4.600000   
               p25     5.110000    2.640719        22.450000   12.000000   
               p50     8.840000    9.245312        31.400000   24.700000   
               p75    13.780000   25.800000        37.950000   38.300000   
               p90    20.290000   41.770967        44.720000   61.200000   
               p95    25.970000   51.400000        53.940000   88.875000   

province                 Cavite        Cebu  Davao del Sur  Eastern Samar  \
lat            mean   14.143177   10.326450       7.143752      10.998140   
               min    14.143177   10.326450       7.143752      10.998140   
               max    14.143177   10.326450       7.143752      10.998140   
               p05    14.143177   10.326450       7.143752      10.998140   
               p10    14.143177   10.326450       7.143752      10.998140   
               p25    14.143177   10.326450       7.143752      10.998140   
               p50    14.143177   10.326450       7.143752      10.998140   
               p75    14.143177   10.326450       7.143752      10.998140   
               p90    14.143177   10.326450       7.143752      10.998140   
               p95    14.143177   10.326450       7.143752      10.998140   
lon            mean  120.984941  123.978710     125.625054     125.649680   
               min   120.984941  123.978710     125.625054     125.649680   
               max   120.984941  123.978710     125.625054     125.649680   
               p05   120.984941  123.978710     125.625054     125.649680   
               p10   120.984941  123.978710     125.625054     125.649680   
               p25   120.984941  123.978710     125.625054     125.649680   
               p50   120.984941  123.978710     125.625054     125.649680   
               p75   120.984941  123.978710     125.625054     125.649680   
               p90   120.984941  123.978710     125.625054     125.649680   
               p95   120.984941  123.978710     125.625054     125.649680   
pm25 in µg/m^3 mean   16.1

## **Unique Values for Province**


In [14]:
abt["province"].unique()

<ArrowStringArray>
[              'NCR', 'Negros Occidental',            'Bataan',
             'Rizal',           'Benguet',            'Iloilo',
            'Laguna',   'Camarines Norte',          'Pampanga',
     'Davao del Sur',              'Cebu',     'Eastern Samar',
            'Cavite',             'Capiz']
Length: 14, dtype: str

# Initial Observations and Notable Findings

1. An overwhelming majority of sensors are located within the NCR province.
2. Only stations in Negros Oriental province provide pm10 data, thus it was excluded due to the limited number of observations.
3. Not all Philippine provinces are represented by air quality stations, a total of 14 provinces have coverage for air quality indicators.
4. Some stations have inconsistent reporting periods.